# 02 · 自己实现 PID：30 行代码的工业基线

PID 是真实广告系统里最常用的预算 pacing 控制器。规则一句话：**上个时段花得比计划慢 → 系数 ×1.2；花太快 → ×0.7**。

本篇用官方数据 + threshold replay（出价 ≥ 最低获胜价即赢，对手冻结）验证你实现的 PID。

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/period7_adv0.csv.gz')
BUDGET, TARGET_CPA, NUM_TICK = float(df.budget.iloc[0]), float(df.CPAConstraint.iloc[0]), 48
print(BUDGET, TARGET_CPA)

In [ ]:
class PID:
    """上时段花费速度 vs 均匀计划 → 调 alpha。就这么简单。"""
    def __init__(self, base_alpha=15.0):
        self.alpha = base_alpha
        self.last_remaining = None

    def act(self, tick, remaining_budget):
        if tick > 0 and remaining_budget > 0:
            last_cost = self.last_remaining - remaining_budget
            ratio = last_cost * (NUM_TICK - tick) / remaining_budget
            if ratio < 0.7:   self.alpha *= 1.2   # 花慢了，加价
            elif ratio > 1.1: self.alpha *= 0.7   # 花快了，降价
        self.last_remaining = remaining_budget
        return self.alpha

In [ ]:
def replay(policy, df, budget=BUDGET, target_cpa=TARGET_CPA):
    """threshold replay: 赢 iff bid >= leastWinningCost, 花费 = leastWinningCost.
    conversions 用期望口径（pValue 求和），低方差、可复现。"""
    remaining, cost_total, conv_total = budget, 0.0, 0.0
    alphas = []
    for tick, g in df.sort_values('timeStepIndex').groupby('timeStepIndex'):
        if remaining < 0.1: 
            alphas.append(0); continue
        alpha = policy.act(tick, remaining)
        alphas.append(alpha)
        bids = alpha * g.pValue.values
        win = bids >= g.leastWinningCost.values
        tick_cost = g.leastWinningCost.values[win].sum()
        if tick_cost > remaining:   # 简化的预算截断
            order = np.cumsum(g.leastWinningCost.values * win)
            win &= order <= remaining
            tick_cost = g.leastWinningCost.values[win].sum()
        remaining -= tick_cost
        cost_total += tick_cost
        conv_total += g.pValue.values[win].sum()
    cpa = cost_total / max(conv_total, 1e-9)
    score = conv_total if cpa <= target_cpa else conv_total * (target_cpa/cpa)**2
    return dict(score=round(score,2), conv=round(conv_total,2), cost=round(cost_total,1),
                cpa=round(cpa,1), util=f'{cost_total/budget:.0%}'), alphas

result, alphas = replay(PID(), df)
result

## 练习

1. 把 `base_alpha` 从 15 改成 60/100/150，score 怎么变？为什么官方市场上"起步价"很重要？
2. PID 只看"花钱速度"不看 CPA——给它加一条"CPA 超标就降价"的规则，score 能提高吗？
3. 对照 01 篇反推的官方 alpha 序列，你的 PID 和数据里那个广告主像吗？